# 32 (PW) — Landing & Ingest

**Production workflow, step 1.** Bring data into IRIS: read CSV/Parquet and a JDBC foreign table, then write a clean landing table. Grounded in `docs/data_access.md` and `docs/foreign_tables_guide.md`.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Read a local CSV

`session.read.csv` infers schema. A local file is read through the client's Arrow bridge.

In [ ]:
import os
import pandas as pd

# Small sample written to a local path the kernel can read.
local = "data/pw_orders.csv"
os.makedirs(os.path.dirname(local), exist_ok=True)
pd.DataFrame({
    "pedido_id": [1, 2, 3],
    "cliente_id": [10, 20, 30],
    "valor": [100.0, 250.0, 90.0],
}).to_csv(local, index=False)

df_csv = session.read.csv(local)
df_csv.show()
print("rows:", df_csv.count())

## 2. Read a Parquet file

Parquet reads via pyarrow; schema and column types come from the file metadata.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

local_pq = "data/pw_orders.parquet"
pq.write_table(pa.Table.from_pandas(pd.DataFrame({
    "pedido_id": [4, 5, 6],
    "cliente_id": [40, 50, 60],
    "valor": [310.0, 120.0, 75.0],
})), local_pq)

df_pq = session.read.parquet(local_pq)
df_pq.show()
print("rows:", df_pq.count())

## 3. Read a JDBC foreign table (data lives elsewhere)

With `foreign=True` IRIS owns the file connection via a foreign table — rows are not copied into Python. (JDBC federation is shown in notebook 35.)

In [ ]:
# A CSV foreign table lets IRIS read the file server-side.
# server_path tells IRIS where the shared volume appears on its side.
try:
    ft = session.read.csv(
        "data/pw_orders.csv",
        foreign=True,
        server_path="/irispark-data/pw_orders.csv",
        options={"header": True},
    )
    ft.show()
    print("foreign read ok")
except Exception as e:
    print("foreign read not available from this path:", str(e)[:120])

## 4. Land the data

Union the two sources and persist a landing table. `mode('overwrite')` keeps re-runs idempotent.

In [ ]:
from irispark.functions import col

landing = df_csv.select("pedido_id", "cliente_id", "valor") \
                .union(df_pq.select("pedido_id", "cliente_id", "valor")) \
                .dropDuplicates(["pedido_id"])
landing.write.mode("overwrite").saveAsTable("pw_landing")
session.table("pw_landing").orderBy("pedido_id").show()
print("landed rows:", session.table("pw_landing").count())

## 5. Ingestion contract

Idempotent, deduplicated, and queryable: `pw_landing` is the stable input for step 2 (Clean & Transform).

In [ ]:
print("landing table columns:", session.table("pw_landing").columns)
print("row count:", session.table("pw_landing").count())

In [ ]:
session.sql("DROP TABLE IF EXISTS pw_landing")
print("dropped pw_landing")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")